<a href="https://colab.research.google.com/github/vladosG005/intro-ai/blob/main/lab2_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №2 Задача классификации

## Задание
Решите пожалуйста следующую задачу.

Используйте классификатор SVM для данных находящихся пo
[ссылке](https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view?usp=share_link) (данный датасет применялся в лабораторной работе №1). Здесь - выходящей меткой (учителем) является столбец / атрибут - `outcome`. Предоставьте результаты, используя следующие ядерные функции:
- Линейное ядро
- Полиномиальное ядро
- Гауссово ядро
- Сигмоидное ядро

Укажите `score` моделей. Сопоставьте полученные результаты и сделайте вывод.

## Загрузка данных

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
! gdown --fuzzy 'https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view?usp=share_link'

Downloading...
From: https://drive.google.com/uc?id=1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec
To: /content/diabetes.csv
100% 23.1k/23.1k [00:00<00:00, 49.8MB/s]


In [3]:
df = pd.read_csv('/content/diabetes.csv')
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


##  О датасете
[Pima Indians Diabetes Database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database/data)

Этот набор данных взят из Национального института диабета, заболеваний органов пищеварения и почек (NIDDK). Целью набора данных является диагностическое предсказание наличия у пациента диабета на основе определенных диагностических измерений, включенных в набор данных. На выбор этих экземпляров из более крупной базы данных было наложено несколько ограничений. В частности, все пациенты здесь — женщины не моложе 21 года, родом из индейцев Пима.

Наборы данных состоят из нескольких переменных медицинских предикторов и одной целевой переменной `Outcome`. Переменные-предикторы включают количество беременностей, которые были у пациентки, её индекс массы тела `BMI`, уровень инсулина, возраст и так далее. Всего в датасете 9 переменных:
- `Pregnancies` - количество беременностей;
- `Glucose` - концентрация глюкозы в плазме через 2 часа при пероральном тесте на толерантность к глюкозе;
- `BloodPressure` - диастолическое артериальное давление (мм рт. ст.);
- `SkinThickness` - толщина кожной складки трицепса (мм);
- `Insulin` - 2-часовой сывороточный инсулин (мкЕд/мл);
- `BMI` - Индекс массы тела (вес в кг/(рост в м)^2);
- `DiabetesPedigreeFunction` - функция оценки вероятности диабета на основе семейного анамнеза;
- `Age` - возраст (лет);
- `Outcome` - целевая переменная (0 или 1) 268 из 768 — 1 (больны), остальные — 0 (здоровы).



In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


In [6]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

## Предобработка данных

In [14]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X.shape, y.shape

((768, 8), (768,))

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=55)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((614, 8), (154, 8), (614,), (154,))

In [18]:
from sklearn.impute import SimpleImputer

zero_filler = SimpleImputer(missing_values=0, strategy="median")
X_train[['Glucose', 'BloodPressure', 'SkinThickness',
         'Insulin', 'BMI']] = zero_filler.fit_transform(X_train[['Glucose', 'BloodPressure', 'SkinThickness',
         'Insulin', 'BMI']])
X_train.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,3.781759,120.693811,72.197068,28.975570,135.923453,32.442671,0.466415,33.273616
std,3.273991,30.378108,12.048783,8.931371,84.835080,6.989666,0.313385,11.767877
min,0.000000,44.000000,24.000000,7.000000,14.000000,18.200000,0.084000,21.000000
25%,1.000000,99.000000,64.000000,25.000000,120.000000,27.425000,0.241750,24.000000
50%,3.000000,116.000000,72.000000,29.000000,120.000000,32.200000,0.374500,29.000000
75%,6.000000,139.000000,80.000000,32.000000,121.500000,36.575000,0.629750,41.000000
max,14.000000,198.000000,122.000000,99.000000,846.000000,67.100000,2.420000,72.000000


In [19]:
X_test[['Glucose', 'BloodPressure', 'SkinThickness',
         'Insulin', 'BMI']] = zero_filler.transform(X_test[['Glucose', 'BloodPressure', 'SkinThickness',
         'Insulin', 'BMI']])

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train[X_train.columns] = scaler.fit_transform(X_train)
X_train.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
count,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02
mean,1.663526e-17,-1.504406e-16,-5.439008e-16,-1.157236e-16,-7.811341e-17,2.762900e-16,-5.496870e-17,-1.677992e-16
std,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00
min,-1.156033e+00,-2.526699e+00,-4.003422e+00,-2.462498e+00,-1.438354e+00,-2.039337e+00,-1.221267e+00,-1.043827e+00
25%,-8.503467e-01,-7.147087e-01,-6.808780e-01,-4.454872e-01,-1.878520e-01,-7.184552e-01,-7.174824e-01,-7.886874e-01
50%,-2.389733e-01,-1.546389e-01,-1.636921e-02,2.737529e-03,-1.878520e-01,-3.474685e-02,-2.935372e-01,-3.634555e-01
75%,6.780868e-01,6.031026e-01,6.481396e-01,3.389061e-01,-1.701562e-01,5.916875e-01,5.216193e-01,6.571010e-01
max,3.123580e+00,2.546874e+00,4.136811e+00,7.846670e+00,8.376906e+00,4.962409e+00,6.238892e+00,3.293539e+00


In [28]:
X_test[X_test.columns] = scaler.transform(X_test)

## Классификатор SVM

### Линейное ядро

In [29]:
from sklearn.svm import SVC
from sklearn.model_selection import KFold, cross_val_score

lin = SVC(kernel='linear', random_state=55)
kf = KFold(n_splits=5, shuffle=True, random_state=55)

cvs = cross_val_score(lin, X_train, y_train, cv=kf)
print(cvs)
print(cvs.mean())

[0.78861789 0.71544715 0.79674797 0.77235772 0.80327869]
0.7752898840463814


In [30]:
lin.fit(X_train, y_train)

lin.score(X_test, y_test)

0.7532467532467533

### Полиномиальное ядро

In [31]:
poly = SVC(kernel='poly', random_state=55)

cvs = cross_val_score(poly, X_train, y_train, cv=kf)
print(cvs)
print(cvs.mean())

[0.7398374  0.70731707 0.76422764 0.75609756 0.7295082 ]
0.7393975743036119


In [32]:
poly.fit(X_train, y_train)

poly.score(X_test, y_test)

0.7077922077922078

### Гауссово ядро

In [33]:
gauss = SVC(kernel='rbf', random_state=55)

cvs = cross_val_score(gauss, X_train, y_train, cv=kf)
print(cvs)
print(cvs.mean())

[0.78861789 0.73170732 0.78861789 0.74796748 0.7704918 ]
0.765480474476876


In [34]:
gauss.fit(X_train, y_train)

gauss.score(X_test, y_test)

0.7467532467532467

### Сигмоидное ядро

In [35]:
sigma = SVC(kernel='sigmoid', random_state=55)

cvs = cross_val_score(sigma, X_train, y_train, cv=kf)
print(cvs)
print(cvs.mean())

[0.72357724 0.69105691 0.72357724 0.7398374  0.77868852]
0.7313474610155938


In [36]:
sigma.fit(X_train, y_train)

sigma.score(X_test, y_test)

0.6883116883116883

## Вывод

SVM с линейным и гауссовым ядром показали себя лучше всего.